In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

## Configuración de rutas y columnas

In [2]:
data_folder = r"C:\Users\erik-\OneDrive\Escritorio\UPIIT\proyecto_actualizaciones"

# Archivos de datos UNSW-NB15
files = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv",
]

# Leer nombres de columnas desde el archivo de features
features_file = os.path.join(data_folder, "NUSW-NB15_features.csv")
FEATURE_NAMES = (
    pd.read_csv(features_file, encoding='latin-1')["Name"]
    .str.strip()
    .tolist()
)

print(f"Columnas cargadas: {len(FEATURE_NAMES)}")

Columnas cargadas: 49


## Features seleccionadas

Basadas en el análisis de correlación con `Label`.
Se descartaron variables redundantes entre sí (ej. `swin`/`dwin`, `Sjit`/`Djit`, `stcpb`/`dtcpb`).

In [3]:
# Ordenadas de mayor a menor correlación con Label:
# sttl (0.837), ct_state_ttl (0.743), dttl (0.454),
# tcprtt (0.331), ackdat (0.329), synack (0.296),
# Sload (0.254), dmeansz (0.137), Dload (0.120),
# swin (0.111), ct_dst_sport_ltm (0.110), Sjit (0.075)

selected_features = [
    'dur',               # duración del flujo
    'sbytes',            # bytes enviados
    'dbytes',            # bytes recibidos
    'sttl',              # TTL fuente          | corr=0.837
    'dttl',              # TTL destino         | corr=0.454
    'Sload',             # bits/s fuente       | corr=0.254
    'Dload',             # bits/s destino      | corr=0.120
    'swin',              # ventana TCP fuente  | corr=0.111
    'smeansz',           # tamaño medio paquete fuente
    'dmeansz',           # tamaño medio paquete destino | corr=0.137
    'Sjit',              # jitter fuente       | corr=0.075
    'Sintpkt',           # tiempo entre paquetes fuente
    'tcprtt',            # RTT TCP             | corr=0.331
    'ackdat',            # tiempo ACK-DATA     | corr=0.329
    'ct_state_ttl',      # conteo estado+TTL   | corr=0.743
    'ct_srv_dst',        # conteo conexiones al mismo servicio/destino
    'ct_dst_sport_ltm',  # conteo conexiones al mismo dst+sport | corr=0.110
    'ct_dst_src_ltm',    # conteo conexiones entre mismo src-dst | corr=0.054
]

print(f"Features seleccionadas: {len(selected_features)}")

Features seleccionadas: 18


## Función de preprocesamiento

In [8]:
def preprocess_unsw_horizonte(file_path, horizonte_pasos=5):

    # Leer el archivo CSV usando los nombres de columnas globales
    df = pd.read_csv(
        file_path,
        header=None,
        names=FEATURE_NAMES,
        encoding='latin-1',
        low_memory=False
    )

    # ── Timestamp real (Unix epoch → datetime) ───────────────────────────
    df["time"] = pd.to_datetime(df["Stime"], unit="s")
    df = df.sort_values("time").reset_index(drop=True)
    df = df.set_index("time")

    # ── Seleccionar features + etiquetas indispensables ──────────────────
    cols_needed = selected_features + ["Label", "attack_cat"]
    df = df[cols_needed].copy()

    # Limpieza básica de la columna categórica
    df["attack_cat"] = df["attack_cat"].astype(str).str.strip().replace({'nan': 'Normal', '': 'Normal'})

    # ── Limpiar infinitos y NaN en características numéricas ─────────────
    df[selected_features] = (
        df[selected_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # ── Agregación por ventana de 1 segundo ──────────────────────────────
    agg_dict = {feat: "mean" for feat in selected_features}
    agg_dict["dur"]    = "mean"
    agg_dict["sbytes"] = "sum"
    agg_dict["dbytes"] = "sum"

    ts_features = df[selected_features].resample("1s").agg(agg_dict)

    # Conteo de flujos por ventana (métrica de volumen por segundo)
    ts_features["flow_count"] = df["Label"].resample("1s").size()

    # Etiqueta en el segundo exacto 't' (1 si algún flujo fue ataque)
    ts_label_actual = df["Label"].resample("1s").max().fillna(0).astype(int)

    # ── CRÍTICO: CREACIÓN DEL HORIZONTE PREDICTIVO (ALERTA TEMPRANA) ─────
    # Revisa si en los próximos 'horizonte_pasos' hay algún ataque (Label=1).
    # Desplazamos negativamente para que el segundo 't' tenga el valor de lo que pasará en el futuro.
    ts_label_futura = ts_label_actual.rolling(window=horizonte_pasos).max().shift(-horizonte_pasos)

    # Categoría dominante de ataque en la ventana
    ts_attack_cat = df["attack_cat"].resample("1s").apply(
        lambda x: x.mode().iloc[0] if not x.mode().empty else "Normal"
    )

    # ── Consolidación del DataFrame indexado por segundo ─────────────────
    ts = ts_features.copy()
    ts["Label_Actual"] = ts_label_actual
    ts["Label"]        = ts_label_futura  # El modelo entrenará para predecir ESTA variable objetivo
    ts["attack_cat"]   = ts_attack_cat
    
    # Eliminamos las últimas N ventanas ya que no tienen "futuro" que predecir (quedan en NaN)
    ts = ts.dropna(subset=["Label"]).fillna(0)
    ts["Label"] = ts["Label"].astype(int)

    return ts

## Procesamiento y guardado de archivos

In [9]:
import os
from sklearn.model_selection import train_test_split

# Definir la carpeta de salida destino acordada
processed_folder = r"C:\Users\erik-\OneDrive\Documentos\GitHub\DETECCION-DE-ANOMALIAS\processed_data"
os.makedirs(processed_folder, exist_ok=True)

# Lista para almacenar los bloques procesados cronológicamente
all_ts_windows = []

# Configuración del horizonte: 5 pasos (segundos) hacia adelante
HORIZONTE_N = 5

print(f"=== FASE 1: Procesamiento por Archivo (Horizonte: {HORIZONTE_N}s) ===")
for file in files:
    path = os.path.join(data_folder, file)
    print(f"Procesando de forma continua: {file} ...", end=" ")
    
    # Procesar manteniendo el orden temporal intacto dentro del archivo
    ts = preprocess_unsw_horizonte(path, horizonte_pasos=HORIZONTE_N)
    all_ts_windows.append(ts)
    
    alertas   = (ts["Label"] == 1).sum()
    normales  = (ts["Label"] == 0).sum()
    pct       = 100 * alertas / len(ts) if len(ts) > 0 else 0
    print(f"OK → {len(ts):,} ventanas | Seguros: {normales:,} | Alertas Futuras: {alertas:,} ({pct:.2f}%)")

# === FASE 2: Consolidación Total del Repositorio ===
print("\n=== FASE 2: Consolidación del Dataset Completo ===")
df_total = pd.concat(all_ts_windows).sort_index()
print(f"Total de ventanas de 1 segundo unificadas: {len(df_total):,}")

# === FASE 3: Partición Estratificada y Rompimiento del Sesgo Global ===
print("\n=== FASE 3: Generación de Particiones Balanceadas (Shuffle Estratificado) ===")
# Mezclamos las ventanas ya etiquetadas con su futuro para que TRAIN y TEST posean idéntica distribución
train_df, test_df = train_test_split(
    df_total,
    test_size=0.3,              # 70% entrenamiento, 30% evaluación externa
    stratify=df_total["Label"],  # Forzar misma proporción de alertas futuras en ambos bloques
    random_state=42,
    shuffle=True                # Destruye el sesgo del tramo horario de la simulación
)

# === FASE 4: Escritura en Disco en la Carpeta de GitHub ===
train_path = os.path.join(processed_folder, "unsw_train_stratified.csv")
test_path = os.path.join(processed_folder, "unsw_test_stratified.csv")

train_df.to_csv(train_path)
test_df.to_csv(test_path)

print("\n=== PROCESO PREDICTIVO FINALIZADO CON ÉXITO ===")
print(f"➔ Conjunto de ENTRENAMIENTO (Train) guardado en: {train_path}")
print(f"   Shape: {train_df.shape} | Alertas futuras: {(train_df['Label'] == 1).sum():,} ({100*(train_df['Label'] == 1).sum()/len(train_df):.2f}%)")
print(f"➔ Conjunto de PRUEBA (Test) guardado en: {test_path}")
print(f"   Shape: {test_df.shape}  | Alertas futuras: {(test_df['Label'] == 1).sum():,} ({100*(test_df['Label'] == 1).sum()/len(test_df):.2f}%)")

=== FASE 1: Procesamiento por Archivo (Horizonte: 5s) ===
Procesando de forma continua: UNSW-NB15_1.csv ... OK → 28,461 ventanas | Seguros: 21,462 | Alertas Futuras: 6,999 (24.59%)
Procesando de forma continua: UNSW-NB15_2.csv ... OK → 2,275,330 ventanas | Seguros: 2,265,191 | Alertas Futuras: 10,139 (0.45%)
Procesando de forma continua: UNSW-NB15_3.csv ... OK → 18,906 ventanas | Seguros: 91 | Alertas Futuras: 18,815 (99.52%)
Procesando de forma continua: UNSW-NB15_4.csv ... OK → 12,080 ventanas | Seguros: 311 | Alertas Futuras: 11,769 (97.43%)

=== FASE 2: Consolidación del Dataset Completo ===
Total de ventanas de 1 segundo unificadas: 2,334,777

=== FASE 3: Generación de Particiones Balanceadas (Shuffle Estratificado) ===

=== PROCESO PREDICTIVO FINALIZADO CON ÉXITO ===
➔ Conjunto de ENTRENAMIENTO (Train) guardado en: C:\Users\erik-\OneDrive\Documentos\GitHub\DETECCION-DE-ANOMALIAS\processed_data\unsw_train_stratified.csv
   Shape: (1634343, 22) | Alertas futuras: 33,405 (2.04%)
➔ C

## Verificación del resultado

In [10]:
print("=== VERIFICACIÓN DE INTEGRIDAD PREDICTIVA ===")
print("Distribución del objetivo (Label = Ataque en los próximos N segundos) en TRAIN:")
print(train_df["Label"].value_counts().rename({0: "Estable (No ataques cerca)", 1: "Peligro (Ataque inminente)"}))

print("\n" + "="*60 + "\n")

print("Distribución del objetivo (Label = Ataque en los próximos N segundos) en TEST:")
print(test_df["Label"].value_counts().rename({0: "Estable (No ataques cerca)", 1: "Peligro (Ataque inminente)"}))
print(test_df["attack_cat"].value_counts().head(5))

=== VERIFICACIÓN DE INTEGRIDAD PREDICTIVA ===
Distribución del objetivo (Label = Ataque en los próximos N segundos) en TRAIN:
Label
Estable (No ataques cerca)    1600938
Peligro (Ataque inminente)      33405
Name: count, dtype: int64


Distribución del objetivo (Label = Ataque en los próximos N segundos) en TEST:
Label
Estable (No ataques cerca)    686117
Peligro (Ataque inminente)     14317
Name: count, dtype: int64
attack_cat
Normal      699765
Generic        637
Exploits        21
DoS             11
Name: count, dtype: int64
